In [ ]:
# PROBE 1 — environment (run once per session; judge_plan §2.2 step 4). Versions are UNVERIFIED until this prints them.
# Runtime -> Change runtime type -> T4 GPU. gpu_run.sh re-records all of this into bench/results/t4/env.txt on every run.
!nvidia-smi --query-gpu=name,driver_version,compute_cap,memory.total,clocks.max.sm --format=csv
!nvcc --version | tail -2; whoami; id -u
!command -v ncu || ls /usr/local/cuda*/bin/ncu 2>/dev/null; ncu --version 2>/dev/null | head -2 || echo ncu-missing
!command -v nsys || echo nsys-missing; command -v compute-sanitizer || ls /usr/local/cuda/bin/compute-sanitizer || echo sanitizer-missing
!python -c "import torch;print('torch',torch.__version__,'cuda',torch.version.cuda,torch.cuda.get_arch_list())"
# CMakePresets.json is schema v6 -> cmake >= 3.25 (Ubuntu 22.04 apt ships 3.22): check the version, not the binary
!cmake --version 2>/dev/null | awk 'NR==1{split($3,v,"."); exit !(v[1]>3 || (v[1]==3 && v[2]>=25))}' && cmake --version | head -1 || pip install -q 'cmake>=3.25' ninja
!nvidia-smi -q -d CLOCK | head -30


In [ ]:
# PROBE 2 — ncu go/no-go (repeat at the start of EVERY session; judge_plan §2.2 step 5).
# PASS = numeric values for all four metrics. FAIL = ERR_NVGPUCTRPERM -> Runtime -> Disconnect and delete runtime
# -> reconnect T4 -> rerun (<=3x) -> else degraded mode (nsys + CUDA events), recorded by scripts/profile.sh.
# Output is also saved to /content/probe2.txt: paste it into docs/RESEARCH.md "Empirical profiler notes".
open('/content/k.cu', 'w').write(r"""#include <cstdio>
__global__ void k(float* x){ int i=blockIdx.x*blockDim.x+threadIdx.x; x[i]=x[i]*2.f+1.f; }
int main(){ float* d; cudaMalloc(&d,1<<20); k<<<256,256>>>(d); cudaDeviceSynchronize(); printf("ok\n"); return 0; }
""")
!nvcc -O2 -lineinfo -arch=sm_75 /content/k.cu -o /content/k && ncu --metrics sm__cycles_elapsed.avg,dram__bytes.sum,sm__warps_active.avg.pct_of_peak_sustained_active,launch__registers_per_thread /content/k 2>&1 | tail -15 | tee /content/probe2.txt
!ncu --query-metrics 2>/dev/null | grep -E '^(sm__throughput|gpu__time_duration|gpu__compute_memory_throughput|sm__pipe_tensor_op_hmma|smsp__warp_issue_stalled_(long_scoreboard|barrier|mio_throttle))' | head | tee -a /content/probe2.txt
!ncu --list-sections 2>/dev/null | grep -E 'SpeedOfLight|MemoryWorkloadAnalysis|Occupancy|WarpStateStats|LaunchStats|SchedulerStats|ComputeWorkloadAnalysis' | tee -a /content/probe2.txt
!ncu --clock-control base --metrics sm__cycles_elapsed.avg /content/k 2>&1 | grep -iE 'clock|warn' || echo "clock-control base accepted"
!nvidia-smi -lgc 585,585 2>&1 | head -2; nvidia-smi -rgc 2>&1 | head -1
!nsys profile -t cuda -o /content/t /content/k >/dev/null 2>&1 && ls -la /content/t.nsys-rep || echo "nsys unavailable"


In [ ]:
# RUNNER — the only cell re-run per stage (judge_plan §2.2 step 6). Edit BRANCH / STAGE / MODE, press Run.
# The <=40-line "====== FA SUMMARY ======" block at the very end of the output is what the lead reads.
# MODE: 'quick' = build + ctest + sanitizer + gpu-pytest ; 'full' = quick + bench + profile (+ push with --push).
import os
from google.colab import userdata
os.environ['GH_PAT'] = userdata.get('GH_PAT')   # Colab Secret (key icon), Notebook access ON; never printed
BRANCH = 'stage0-scaffold'    # edit per run
STAGE  = '0'                  # edit per run: 0 = smoke only
MODE   = 'quick'              # 'quick' | 'full'
!test -d /content/fa || git clone https://github.com/yashpatil582/fused-attention-cuda.git /content/fa
!cd /content/fa && git fetch -q origin && git checkout -q $BRANCH && git reset -q --hard origin/$BRANCH && bash scripts/gpu_run.sh --stage $STAGE --gpu t4 --mode $MODE --push
